# Instalación de librerías y comprobación de entorno


In [1]:
# Dependencias necesarias — instalar antes de ejecutar este notebook:
#
#   GPU NVIDIA (CUDA 12.4 — RTX 4070 Super / driver >= 525):
#     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#
#   Resto del stack:
#     pip install -r requirements/finetuning.txt

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" | GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("\nSin GPU: el entrenamiento será muy lento. Se recomienda GPU con >= 8 GB VRAM.")

PyTorch 2.6.0+cu124 | CUDA: True | GPU: NVIDIA GeForce RTX 4070 SUPER (13 GB)


In [2]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Asegura que el directorio de trabajo sea la raíz del repositorio
# (necesario si Jupyter se abrió desde src/finetuning/ en vez de desde la raíz)
_here = Path.cwd()
if not (_here / "requirements").exists():
    for _parent in _here.parents:
        if (_parent / "requirements").exists():
            os.chdir(_parent)
            break
    else:
        raise RuntimeError(
            f"No se encontró la raíz del repositorio desde {_here}.\n"
            "Ejecuta Jupyter desde la raíz del proyecto: jupyter lab"
        )

# Importar funciones auxiliares del módulo local
sys.path.insert(0, str(Path.cwd() / "src" / "finetuning"))
from functions import (
    LABEL_RELEVANTE, LABEL_NO_RELEVANTE, LABELS, GRADING_SYSTEM_PROMPT,
    check_gpu, load_grading_dataset, split_dataset, show_split_stats,
    build_grading_messages, format_training_prompt, examples_to_hf_dataset,
    load_tokenizer, load_model_4bit, build_peft_model, get_training_args,
    run_training, save_adapter, predict_relevance, evaluate_model,
    print_comparison, log_experiment,
)

check_gpu()

PyTorch:          2.6.0+cu124
CUDA disponible:  True
GPU:              NVIDIA GeForce RTX 4070 SUPER
VRAM total:       12.9 GB


In [3]:
from pathlib import Path

# El CWD ya fue fijado a la raíz del repositorio en la celda anterior (env-check-05)

# Rutas locales (relativas a la raíz del repositorio)
DATASET_PATH = Path("data/processed/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR   = "src/finetuning/output/qwen-grader-lora"
ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"
MERGED_PATH  = "src/finetuning/output/qwen-grader-merged"

MAX_SEQ_LEN = 512

print(f"Directorio de trabajo: {Path.cwd()}")
print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

Directorio de trabajo: c:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final
Configuración:
  Modelo base:    Qwen/Qwen2.5-3B-Instruct
  Dataset:        data\processed\grading_dataset.jsonl  (existe: True)
  Output dir:     src/finetuning/output/qwen-grader-lora
  Adapter path:   src/finetuning/output/qwen-grader-lora/adapter_final
  Max seq len:    512


In [4]:
all_data = load_grading_dataset(DATASET_PATH)

Dataset cargado: C:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final\data\processed\grading_dataset.jsonl
  Total ejemplos:  634
  Relevantes:      283 (44.6%)
  No relevantes:   351 (55.4%)

Muestra:
{
  "query": "¿Cuándo es obligatorio recurrir a un organismo notificado para evaluar conformidad de IA?",
  "document": "Artículo 1258 del Código Civil — Contratos. Los contratos se perfeccionan por el mero consentimiento, y desde entonces obligan no sólo al cumplimiento de lo expresamente pactado, sino también a todas las consecuencias que, según su naturaleza, sean conformes a la buena fe, al uso y a la ley.",
  "label": "no relevante"
}


In [5]:
train_data, val_data, test_data = split_dataset(all_data)

Split del dataset:
  Train: 443 ejemplos
  Val:   95 ejemplos
  Test:  96 ejemplos


# Evaluación baseline: modelo base sin fine-tunear

Antes de entrenar, evaluamos el rendimiento del modelo base con **prompting directo**.
Esto establece el punto de comparación para medir el impacto del fine-tuning.

El prompt de grading que usamos aquí es equivalente al que ya usa `src/rag/main.py`,
pero adaptado para producir `"relevante"` / `"no relevante"` en lugar de `"si"` / `"no"`.


In [6]:
tokenizer  = load_tokenizer(MODEL_ID)
base_model = load_model_4bit(MODEL_ID)

c:\Users\rammu\anaconda3\envs\venv_finetuning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando tokenizador...
Tokenizador listo. Vocabulario: 151,643 tokens
Cargando Qwen/Qwen2.5-3B-Instruct en 4-bit NF4 (evaluacion (4-bit))...


Loading weights: 100%|██████████| 434/434 [00:05<00:00, 76.73it/s, Materializing param=model.norm.weight]                               


Modelo cargado. Dispositivo: cuda:0


In [7]:
sample = test_data[0]
pred = predict_relevance(sample["query"], sample["document"], base_model, tokenizer)
print("Prediccion baseline (muestra):")
print(f"  Query:    {sample['query'][:80]}...")
print(f"  Predicho: {pred}")
print(f"  Real:     {sample['label']}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prediccion baseline (muestra):
  Query:    ¿Qué pasa con la conformidad de un sistema de IA cuando sufre una modificación s...
  Predicho: no relevante
  Real:     no relevante


In [8]:
baseline_acc, baseline_f1 = evaluate_model(
    test_data, base_model, tokenizer,
    name="BASELINE — Qwen 2.5 3B-Instruct (sin fine-tuning)"
)

Evaluando BASELINE — Qwen 2.5 3B-Instruct (sin fine-tuning) en 96 ejemplos...
  5/96 completados
  10/96 completados
  15/96 completados
  20/96 completados
  25/96 completados
  30/96 completados
  35/96 completados
  40/96 completados
  45/96 completados
  50/96 completados
  55/96 completados
  60/96 completados
  65/96 completados
  70/96 completados
  75/96 completados
  80/96 completados
  85/96 completados
  90/96 completados
  95/96 completados

BASELINE — Qwen 2.5 3B-Instruct (sin fine-tuning)
Accuracy:  0.8542
F1-macro:  0.8500

              precision    recall  f1-score   support

   relevante       0.89      0.77      0.82        43
no relevante       0.83      0.92      0.88        53

    accuracy                           0.85        96
   macro avg       0.86      0.85      0.85        96
weighted avg       0.86      0.85      0.85        96



# Evaluación del modelo fine-tuneado

Evaluamos el modelo con los mismos ejemplos de test que usamos para el baseline,
para poder comparar directamente ambos enfoques.


In [9]:
from pathlib import Path
from peft import PeftModel

# Verificar que el adaptador existe
assert Path(ADAPTER_PATH).exists(), (
    f"Adaptador no encontrado en {Path(ADAPTER_PATH).resolve()}\n"
    "Ejecuta primero 02_entrenamiento.ipynb para generar el adaptador."
)

# Liberar base_model de VRAM antes de cargar el fine-tuneado
try:
    del base_model
    torch.cuda.empty_cache()
    print("Modelo baseline liberado de VRAM.")
except NameError:
    pass

# Cargar modelo base en 4-bit y aplicar el adaptador LoRA guardado
print(f"Cargando adaptador LoRA desde: {ADAPTER_PATH}")
_base = load_model_4bit(MODEL_ID)
model = PeftModel.from_pretrained(_base, ADAPTER_PATH)
model.eval()
print("Modelo fine-tuneado listo.")

Modelo baseline liberado de VRAM.


Cargando adaptador LoRA desde: src/finetuning/output/qwen-grader-lora/adapter_final
Cargando Qwen/Qwen2.5-3B-Instruct en 4-bit NF4 (evaluacion (4-bit))...


Loading weights: 100%|██████████| 434/434 [00:03<00:00, 124.09it/s, Materializing param=model.norm.weight]                              


Modelo cargado. Dispositivo: cuda:0
Modelo fine-tuneado listo.


In [10]:
model.eval()
finetuned_acc, finetuned_f1 = evaluate_model(
    test_data, model, tokenizer,
    name="FINE-TUNED — Qwen 2.5 3B + QLoRA"
)

Evaluando FINE-TUNED — Qwen 2.5 3B + QLoRA en 96 ejemplos...
  5/96 completados
  10/96 completados
  15/96 completados
  20/96 completados
  25/96 completados
  30/96 completados
  35/96 completados
  40/96 completados
  45/96 completados
  50/96 completados
  55/96 completados
  60/96 completados
  65/96 completados
  70/96 completados
  75/96 completados
  80/96 completados
  85/96 completados
  90/96 completados
  95/96 completados

FINE-TUNED — Qwen 2.5 3B + QLoRA
Accuracy:  0.9062
F1-macro:  0.9057

              precision    recall  f1-score   support

   relevante       0.87      0.93      0.90        43
no relevante       0.94      0.89      0.91        53

    accuracy                           0.91        96
   macro avg       0.90      0.91      0.91        96
weighted avg       0.91      0.91      0.91        96



In [ ]:
import pandas as pd

# ── Comparativa global ────────────────────────────────────────────────────────
mejora_acc, mejora_f1 = print_comparison(
    baseline_acc, baseline_f1, finetuned_acc, finetuned_f1
)

# ── Comparativa por clase ─────────────────────────────────────────────────────
# Valores extraídos de los classification_report impresos en las celdas
# baseline-eval-14 y eval-ft-24 (test set: 96 ejemplos, 43 rel / 53 no-rel).
per_class = pd.DataFrame({
    "Modelo":    ["Baseline", "Baseline", "Fine-tuned", "Fine-tuned"],
    "Clase":     ["relevante", "no relevante", "relevante", "no relevante"],
    "Precision": [0.89, 0.83, 0.87, 0.94],
    "Recall":    [0.77, 0.92, 0.93, 0.89],
    "F1":        [0.82, 0.88, 0.90, 0.91],
    "Support":   [43,   53,   43,   53],
})

print("\nComparativa por clase:")
print(per_class.to_string(index=False))

# ── Delta (fine-tuned − baseline) ─────────────────────────────────────────────
delta = pd.DataFrame({
    "Clase":       ["relevante",        "no relevante"],
    "ΔPrecision":  [0.87 - 0.89,        0.94 - 0.83],
    "ΔRecall":     [0.93 - 0.77,        0.89 - 0.92],
    "ΔF1":         [0.90 - 0.82,        0.91 - 0.88],
})

print("\nDelta por clase (fine-tuned − baseline):")
print(delta.to_string(index=False))

print(f"\nResumen: el fine-tuning mejora sobre todo el recall de 'relevante' "
      f"(+{0.93 - 0.77:+.2f}) y la precision de 'no relevante' (+{0.94 - 0.83:+.2f}).")

# Integración con el orquestador (`src/rag/main.py`)

El modelo fine-tuneado reemplaza al modelo base en la función `grade()` de `src/rag/main.py`.
La salida cambia de `"si"`/`"no"` a `"relevante"`/`"no relevante"` para mayor claridad.

## Opciones de integración

| Opción | Cuándo usarla |
|--------|---------------|
| **A — Carga directa PEFT** | Desarrollo, testing, entornos con GPU propia |
| **B — Convertir a GGUF + Ollama** | Producción (mismo setup que el modelo actual) |

El fragmento de código de abajo muestra la **Opción A**.
Para la Opción B, hacer merge del adaptador (celda anterior), convertir con `llama.cpp`,
y actualizar el `model` en `_get_grading_llm()` a `qwen2.5-grader:3b`.


In [12]:
# ============================================================
# Fragmento de integración para src/rag/main.py
# ============================================================
# Copiar este código en src/rag/main.py para activar el grader fine-tuneado.
# Requiere que el adaptador esté en FINETUNED_ADAPTER_PATH.
# ============================================================

INTEGRATION_SNIPPET = '''
# ---- Añadir en src/rag/main.py ----

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

FINETUNED_ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"

GRADING_SYSTEM_PROMPT_FT = (
    "Eres un asistente especializado en normativa de inteligencia artificial. "
    "Tu tarea es determinar si un documento contiene información útil para responder "
    "una consulta sobre regulación de IA (EU AI Act, BOE, normativa española). "
    "Responde únicamente con \'relevante\' o \'no relevante\', sin explicación adicional."
)

_grading_model_ft = None
_grading_tok_ft   = None


def _get_grading_model_finetuned():
    """Carga el grader fine-tuneado (Qwen 2.5 3B + LoRA) — singleton."""
    global _grading_model_ft, _grading_tok_ft
    if _grading_model_ft is None:
        base = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen2.5-3B-Instruct",
            device_map="auto",
            torch_dtype=torch.float16,
        )
        _grading_model_ft = PeftModel.from_pretrained(base, FINETUNED_ADAPTER_PATH)
        _grading_model_ft.eval()
        _grading_tok_ft = AutoTokenizer.from_pretrained(FINETUNED_ADAPTER_PATH)
    return _grading_model_ft, _grading_tok_ft


def grade_with_finetuned(query: str, docs: list[dict], threshold: float = 0.7) -> list[dict]:
    """
    Versión actualizada de grade() que usa el modelo fine-tuneado.
    Salida: "relevante" / "no relevante" (en lugar de "si" / "no").
    Fallback a filtro por score si el modelo no está disponible.
    """
    if not docs:
        return []

    try:
        model_ft, tok_ft = _get_grading_model_finetuned()
    except Exception:
        logger.warning("Grader fine-tuneado no disponible, usando fallback por score")
        return _grade_by_score(docs, threshold)

    relevant = []
    for doc in docs:
        messages = [
            {"role": "system", "content": GRADING_SYSTEM_PROMPT_FT},
            {"role": "user", "content": (
                f"Consulta: {query}\\n\\n"
                f"Documento: {doc[\'doc\']}\\n\\n"
                "¿Es este documento relevante para responder la consulta?"
            )},
        ]
        text   = tok_ft.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok_ft(text, return_tensors="pt").to(model_ft.device)

        with torch.no_grad():
            out = model_ft.generate(**inputs, max_new_tokens=12, do_sample=False,
                                    pad_token_id=tok_ft.eos_token_id)
        gen      = out[0][inputs["input_ids"].shape[1]:]
        response = tok_ft.decode(gen, skip_special_tokens=True).strip().lower()

        if "no relevante" not in response and "relevante" in response:
            relevant.append(doc)

    return relevant
'''

print(INTEGRATION_SNIPPET)


# ---- Añadir en src/rag/main.py ----

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

FINETUNED_ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"

GRADING_SYSTEM_PROMPT_FT = (
    "Eres un asistente especializado en normativa de inteligencia artificial. "
    "Tu tarea es determinar si un documento contiene información útil para responder "
    "una consulta sobre regulación de IA (EU AI Act, BOE, normativa española). "
    "Responde únicamente con 'relevante' o 'no relevante', sin explicación adicional."
)

_grading_model_ft = None
_grading_tok_ft   = None


def _get_grading_model_finetuned():
    """Carga el grader fine-tuneado (Qwen 2.5 3B + LoRA) — singleton."""
    global _grading_model_ft, _grading_tok_ft
    if _grading_model_ft is None:
        base = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen2.5-3B-Instruct",
            device_map="auto",
            torch_dtype=torch.float16,
      

In [13]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

print(f"Verificando adaptador en: {ADAPTER_PATH}")

# Liberar el modelo de evaluación antes de recargar para verificar el adaptador
try:
    del model
    torch.cuda.empty_cache()
except NameError:
    pass

base_for_test = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)
loaded_model = PeftModel.from_pretrained(base_for_test, ADAPTER_PATH)
loaded_model.eval()
loaded_tok = load_tokenizer(ADAPTER_PATH)

print("\nPruebas de inferencia con modelo recargado desde disco:")
for ex in [test_data[0], test_data[-1]]:
    pred = predict_relevance(ex["query"], ex["document"], loaded_model, loaded_tok)
    ok = pred == ex["label"]
    print(f"  {'OK' if ok else 'FAIL'}  Real: {ex['label']:<15} | Predicho: {pred}")

print("\nAdaptador cargado y funcionando correctamente.")

`torch_dtype` is deprecated! Use `dtype` instead!


Verificando adaptador en: src/finetuning/output/qwen-grader-lora/adapter_final


Loading weights: 100%|██████████| 434/434 [00:06<00:00, 70.76it/s, Materializing param=model.norm.weight]                               


Cargando tokenizador...
Tokenizador listo. Vocabulario: 151,643 tokens

Pruebas de inferencia con modelo recargado desde disco:
  OK  Real: no relevante    | Predicho: no relevante
  OK  Real: relevante       | Predicho: relevante

Adaptador cargado y funcionando correctamente.
